# 01 — Coleta e leitura dos dados MapBiomas Monitor do Fogo

Coleta de **área queimada mensal em hectares** para Brasil, estados (incluindo o Distrito Federal) e biomas, no período de **dezembro de 2024 a novembro de 2026**.

**Fonte:** [documentação da API](https://plataforma.monitorfogo.mapbiomas.org/api/docs/#/) e [descrição do Monitor do Fogo](https://brasil.mapbiomas.org/iniciativas-e-produtos/fogo/monitor-do-fogo/).



## Por que coletar as estatísticas?

As estatísticas são adequadas ao objetivo do README porque acrescentam a **extensão da área atingida pelo fogo** às variáveis do INPE. Focos de calor, FRP e área queimada medem aspectos diferentes: os valores em hectares não representam contagem de focos nem intensidade do fogo.

| Recurso da API | Utilidade para este projeto |
| --- | --- |
| `statistics/time-series` | Escolha principal: série mensal para comparar os ciclos e relacionar com INPE e RONI |
| `statistics/years` e `statistics/{year}/months` | Conferir a cobertura e identificar meses ainda indisponíveis |
| `statistics/ranking/.../raw` | Obter códigos e nomes de todos os estados e biomas, sem paginação |
| `statistics/area` | Resumos de área e variações; a série mensal permite calcular os recortes dezembro–novembro |
| `statistics/land-cover` | Possível ampliação para estudar vegetação, pastagem e outras classes de uso do solo |
| Mapas e tiles | Úteis para análise espacial; não são necessários para esta primeira coleta tabular |

As tabelas por estado e por bioma são recortes separados. Elas não representam o cruzamento estado × bioma. O total do Brasil também é coletado diretamente: não some as três tabelas entre si.


## 1. Bibliotecas e pasta de dados


In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen

import pandas as pd

DATA_DIR = Path("../dados")
DATA_DIR.mkdir(parents=True, exist_ok=True)

API_URL = "https://plataforma.monitorfogo.mapbiomas.org/api"
INICIO = "2024-12"
FIM = "2026-11"
ARQUIVO_JSON = DATA_DIR / "mapbiomas_respostas_api.json"
respostas = {}
coleta_iniciada_em = datetime.now(timezone.utc).isoformat()


## 2. Consultar a API e conferir os meses disponíveis

Esta pequena função lê o JSON e guarda a resposta e sua URL para salvar a fonte original ao final. 

Os meses sem publicação são registrados como indisponíveis, sem receber área zero.


In [2]:
def consultar_api(caminho, **parametros):
    url = f"{API_URL}/{caminho}"
    if parametros:
        url += "?" + urlencode(parametros)
    with urlopen(url, timeout=60) as resposta:
        dados = json.load(resposta)
    respostas[url] = dados
    return dados


anos_disponiveis = consultar_api("statistics/years")
meses_disponiveis = {}
for ano in range(int(INICIO[:4]), int(FIM[:4]) + 1):
    meses_disponiveis[ano] = (
        consultar_api(f"statistics/{ano}/months") if ano in anos_disponiveis else []
    )

cobertura = pd.DataFrame({"ano_mes": pd.period_range(INICIO, FIM, freq="M").astype(str)})
cobertura["disponivel"] = [
    int(data[5:7]) in meses_disponiveis[int(data[:4])] for data in cobertura["ano_mes"]
]
print("Meses indisponíveis:", cobertura.loc[~cobertura["disponivel"], "ano_mes"].tolist())
cobertura


Meses indisponíveis: ['2026-11']


,ano_mes,disponivel
0,2024-12,True
1,2025-01,True
2,2025-02,True
3,2025-03,True
4,2025-04,True
5,2025-05,True
6,2025-06,True
7,2025-07,True
8,2025-08,True
9,2025-09,True


## 3. Identificar Brasil, estados e biomas

Conferimos se o catálogo retorna as 27 unidades federativas e os seis biomas.


In [3]:
busca = consultar_api("territories/search/Brasil")
brasil = next(item for item in busca if item["type"] == "country" and item["name"] == "Brasil")
codigo_brasil = brasil["code"]
territorios = [{"tipo": "country", "codigo": codigo_brasil, "nome": "Brasil", "agrupamento": "state"}]

for tipo, quantidade in [("state", 27), ("biome", 6)]:
    catalogo = consultar_api(
        f"statistics/ranking/country/{codigo_brasil}/{tipo}/raw",
        year=2025, monthStart=1, monthEnd=12,
    )
    if len({item["code"] for item in catalogo}) != quantidade:
        raise ValueError(f"Catálogo incompleto para {tipo}; confira a resposta da API.")
    territorios.extend(
        {"tipo": tipo, "codigo": item["code"], "nome": item["name"], "agrupamento": tipo}
        for item in catalogo
    )

print(f"Territórios para coleta: {len(territorios)}")
pd.DataFrame(territorios).head()


Territórios para coleta: 34


,tipo,codigo,nome,agrupamento
0,country,0,Brasil,state
1,state,17,Tocantins,state
2,state,21,Maranhão,state
3,state,51,Mato Grosso,state
4,state,22,Piauí,state


## 4. Coletar as séries mensais

É feita uma consulta por território. A API retorna a série histórica;


In [4]:
tabelas = []
for territorio in territorios:
    tipo = territorio["tipo"]
    codigo = territorio["codigo"]
    agrupamento = territorio["agrupamento"]
    serie = consultar_api(
        f"statistics/time-series/{tipo}/{codigo}/{agrupamento}",
        monthStart=1, monthEnd=12,
    )
    tabela = pd.DataFrame(serie["monthly"])
    tabela["tipo_territorio"] = tipo
    tabela["codigo_territorio"] = codigo
    tabela["territorio"] = territorio["nome"]
    tabelas.append(tabela)
    print(f"Coletado: {territorio['nome']}")

df = pd.concat(tabelas, ignore_index=True)


Coletado: Brasil
Coletado: Tocantins
Coletado: Maranhão
Coletado: Mato Grosso
Coletado: Piauí
Coletado: Pará
Coletado: Roraima
Coletado: Bahia
Coletado: Goiás
Coletado: Amazonas
Coletado: Minas Gerais
Coletado: Amapá
Coletado: Mato Grosso do Sul
Coletado: Rondônia
Coletado: Acre
Coletado: São Paulo
Coletado: Ceará
Coletado: Rio Grande do Sul
Coletado: Paraná
Coletado: Rio de Janeiro
Coletado: Distrito Federal
Coletado: Pernambuco
Coletado: Santa Catarina
Coletado: Paraíba
Coletado: Rio Grande do Norte
Coletado: Alagoas
Coletado: Espírito Santo
Coletado: Sergipe
Coletado: Cerrado
Coletado: Amazônia
Coletado: Caatinga
Coletado: Mata Atlântica
Coletado: Pantanal
Coletado: Pampa


## 5. Organizar os dados e recortar os ciclos

`year`, `month` e `areaHa` são renomeados para `ano`, `mes` e `area_queimada_ha`. Cada linha representa um território em um mês. `ano_mes` facilita a integração com os dados mensais do INPE;

In [5]:
df = df.rename(columns={"year": "ano", "month": "mes", "areaHa": "area_queimada_ha"})
df["ano_mes"] = pd.to_datetime(dict(year=df["ano"], month=df["mes"], day=1)).dt.strftime("%Y-%m")
meses_publicados = cobertura.loc[cobertura["disponivel"], "ano_mes"]
df = df.loc[df["ano_mes"].isin(meses_publicados)].copy()

chave = ["tipo_territorio", "codigo_territorio", "ano_mes"]
if df.duplicated(chave).any() or len(df) != len(territorios) * len(meses_publicados):
    raise ValueError("Existem meses ausentes ou duplicados nas séries; confira as respostas da API.")
df["area_queimada_ha"] = pd.to_numeric(df["area_queimada_ha"], errors="raise")
if df.empty or df["area_queimada_ha"].isna().any() or (df["area_queimada_ha"] < 0).any():
    raise ValueError("A coleta está vazia ou contém áreas inválidas.")

ano_inicio = df["ano"] - (df["mes"] < 12).astype(int)
df["periodo"] = ano_inicio.astype(str) + "–" + (ano_inicio + 1).astype(str)
df["mes_ciclo"] = df["mes"] % 12 + 1
df["tipo_territorio"] = df["tipo_territorio"].replace({"country": "brasil", "state": "estado", "biome": "bioma"})
df = df[["tipo_territorio", "codigo_territorio", "territorio", "ano", "mes", "ano_mes", "periodo", "mes_ciclo", "area_queimada_ha"]]
df = df.sort_values(["tipo_territorio", "codigo_territorio", "ano_mes"]).reset_index(drop=True)
df.head()


,tipo_territorio,codigo_territorio,territorio,ano,mes,ano_mes,periodo,mes_ciclo,area_queimada_ha
0,bioma,1,Amazônia,2024,12,2024-12,2024–2025,1,959838.80
1,bioma,1,Amazônia,2025,1,2025-01,2024–2025,2,629657.39
2,bioma,1,Amazônia,2025,2,2025-02,2024–2025,3,89022.90
3,bioma,1,Amazônia,2025,3,2025-03,2024–2025,4,55112.88
4,bioma,1,Amazônia,2025,4,2025-04,2024–2025,5,30003.98


## 6. Salvar o JSON original e os CSVs

- O JSON reúne as URLs consultadas, as respostas originais (incluindo as séries históricas completas), o horário UTC da coleta e o recorte solicitado. 
- Os CSVs contêm apenas os meses publicados dentro do recorte.
- `mapbiomas_cobertura_mensal.csv` registra a disponibilidade de todos os meses solicitados.


In [6]:
dados_originais = {
    "coleta_iniciada_em_utc": coleta_iniciada_em,
    "coleta_finalizada_em_utc": datetime.now(timezone.utc).isoformat(),
    "inicio": INICIO,
    "fim": FIM,
    "respostas": respostas,
}
ARQUIVO_JSON.write_text(json.dumps(dados_originais, ensure_ascii=False, indent=2), encoding="utf-8")
cobertura.to_csv(DATA_DIR / "mapbiomas_cobertura_mensal.csv", index=False, encoding="utf-8")

for tipo, sufixo in [("brasil", "brasil"), ("estado", "estados"), ("bioma", "biomas")]:
    arquivo = DATA_DIR / f"mapbiomas_area_queimada_{sufixo}.csv"
    tabela = df.loc[df["tipo_territorio"] == tipo]
    tabela.to_csv(arquivo, index=False, encoding="utf-8")
    print(f"Salvo: {arquivo} — {len(tabela)} registros")

print(f"Respostas originais: {ARQUIVO_JSON}")


Salvo: ../dados/mapbiomas_area_queimada_brasil.csv — 23 registros
Salvo: ../dados/mapbiomas_area_queimada_estados.csv — 621 registros
Salvo: ../dados/mapbiomas_area_queimada_biomas.csv — 138 registros
Respostas originais: ../dados/mapbiomas_respostas_api.json


## 7. Leitura e exploração inicial


In [7]:
df_brasil = pd.read_csv(DATA_DIR / "mapbiomas_area_queimada_brasil.csv")
df_estados = pd.read_csv(DATA_DIR / "mapbiomas_area_queimada_estados.csv")
df_biomas = pd.read_csv(DATA_DIR / "mapbiomas_area_queimada_biomas.csv")

df_brasil.head()


,tipo_territorio,codigo_territorio,territorio,ano,mes,ano_mes,periodo,mes_ciclo,area_queimada_ha
0,brasil,0,Brasil,2024,12,2024-12,2024–2025,1,1097364.98
1,brasil,0,Brasil,2025,1,2025-01,2024–2025,2,684858.71
2,brasil,0,Brasil,2025,2,2025-02,2024–2025,3,119853.85
3,brasil,0,Brasil,2025,3,2025-03,2024–2025,4,111370.20
4,brasil,0,Brasil,2025,4,2025-04,2024–2025,5,69155.78


In [8]:
df_estados.info()


<class 'pandas.DataFrame'>
RangeIndex: 621 entries, 0 to 620
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tipo_territorio    621 non-null    str    
 1   codigo_territorio  621 non-null    int64  
 2   territorio         621 non-null    str    
 3   ano                621 non-null    int64  
 4   mes                621 non-null    int64  
 5   ano_mes            621 non-null    str    
 6   periodo            621 non-null    str    
 7   mes_ciclo          621 non-null    int64  
 8   area_queimada_ha   621 non-null    float64
dtypes: float64(1), int64(4), str(4)
memory usage: 43.8 KB


In [9]:
print("Valores nulos por coluna:")
print(df.isnull().sum())
print("\nQuantidade de meses por ciclo (Brasil):")
print(df_brasil.groupby("periodo")["ano_mes"].nunique())
print("\nÚltimo mês coletado:", df_brasil["ano_mes"].max())


Valores nulos por coluna:
tipo_territorio      0
codigo_territorio    0
territorio           0
ano                  0
mes                  0
ano_mes              0
periodo              0
mes_ciclo            0
area_queimada_ha     0
dtype: int64

Quantidade de meses por ciclo (Brasil):
periodo
2024–2025    12
2025–2026    11
Name: ano_mes, dtype: int64

Último mês coletado: 2026-10
